# Data preparation - Feature Engineering
Created by Guillermo Arredondo Renero

Creation date: May 5, 2026  
Last updated: May 5, 2026

**Main objective**: Realize feature engineering over interim data (cleaned data) to empower models in next stage. **Only for continous prediction - Amount of previous payment in June**

In [1]:
# ---------------------------- Libraries
## Directories
import os
## Data manipulation
import pandas as pd
import numpy as np

## Visualizations
import matplotlib.pyplot as plt
import seaborn as sns
import sidetable

# Extras
import warnings
warnings.filterwarnings('ignore')

# Transformation
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split

In [2]:
# ---------------------------- Directories
# Get main directory
CURRENT_DIR = os.getcwd()
MAIN_DIR = os.path.dirname(CURRENT_DIR)
INTERIM_DIR = os.path.join(MAIN_DIR, 'data', 'interim')
PROCESSED_DIR = os.path.join(MAIN_DIR, 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

## Data loading

In [3]:
df = pd.read_csv(os.path.join(INTERIM_DIR, 'credit_dataset_cleaned.csv'))

In [ ]:
# Drop variables after prediction time 1 = September, 2=August, 3=July and default payment as next month probably refers to october
out_of_time_cols = [col for col in df.columns if col.endswith(['_1', '_2', '_3']) or col == "default payment next month"]
print(f"Dropping {len(out_of_time_cols)} columns related to out-of-time features.")
print(out_of_time_cols)
df.drop(columns=out_of_time_cols, inplace=True)
df.head()

### Categorical variables

Dado que no son muchas categorías para las variables categóricas no tenemos problema en usar One-Hot Encoding.  
Nótese que crearemos las variables dummy, sin embargo, más adelante de acuerdo al tipo de modelo que se vaya a utilizar se seleccionarán las variables ya que pudieramos usar la variable en forma ordinal para árboles de decisión. 

*Nota:* aquí podría usarse una categorización por medio de WOE para generar agrupaciones de categorías similares o para categorizar (a través de bins) las variables continuas.

In [ ]:
# Get Female dummy variable
df['Female'] = df['SEX'] - 1

# Replace marriage and education with mapping
education_map = {1: 'Graduate', 2: 'University', 3: 'High School', 4: 'Others'}
marriage_map = {1: 'Married', 2: 'Single', 3: 'Others'}
df['EDUCATION'] = df['EDUCATION'].map(education_map)
df['MARRIAGE'] = df['MARRIAGE'].map(marriage_map)

# Create One Hot Encodings for MARRIAGE and EDUCATION
df = pd.concat(
    [
        df,
        pd.get_dummies(df.MARRIAGE),
        pd.get_dummies(df.EDUCATION)
    ],
    axis=1
)
df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,PAY_AMT6,default_payment_next_month,Female,Married,Others,Single,Graduate,High School,Others,University
0,20000,2,University,Married,24,3,3,-1,-1,-1,...,0,1,1,True,False,False,False,False,False,True
1,120000,2,University,Single,26,-1,3,1,1,1,...,2000,1,1,False,False,True,False,False,False,True
2,90000,2,University,Single,34,1,1,1,1,1,...,5000,0,1,False,False,True,False,False,False,True
3,50000,2,University,Married,37,1,1,1,1,1,...,1000,0,1,True,False,False,False,False,False,True
4,50000,1,University,Married,57,-1,1,-1,1,1,...,679,0,0,True,False,False,False,False,False,True


### New variables

In [ ]:
# Calculate credit utilization (most recent bill amount / credit limit)
df['credit_utilization'] = df['BILL_AMT1'] / df['LIMIT_BAL']  # BILL_AMT1 is most recent bill amount

# Calculate average monthly bill and payment
bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
pay_amt_cols = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

df['avg_monthly_bill'] = df[bill_cols].mean(axis=1)
df['avg_monthly_payment'] = df[pay_amt_cols].mean(axis=1)

# Create a feature: maximum payment delay across 6 months
df['max_payment_delay'] = df[pay_cols].max(axis=1)
df['has_any_delay'] = (df['max_payment_delay'] > 0).astype(int)

### 

In [ ]:
CONT_PRED_DIR = os.path.join(PROCESSED_DIR, "continuous_prediction")
os.makedirs(CONT_PRED_DIR, exist_ok=True)

### Data partition

In [ ]:
# Create directories for train, test and val sets
for split in ['train', 'test', 'val']:
    os.makedirs(os.path.join(CONT_PRED_DIR, split), exist_ok=True)

In [ ]:
# Split data into features and target
X = df.drop(columns=['default_payment_next_month'])
y = df['default_payment_next_month']
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
# Splint into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train)  # 0.25 x 0.8 = 0.2